In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/lopezjatjat10@gmail.com/sportsbar-etl-proj/1_setup/utilities

In [0]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'customers', 'Data Source')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

In [0]:
df_duplicated = df_bronze.groupBy('customer_id').count().filter(F.col('count')> 1)
display(df_duplicated)

In [0]:
print(f'Row Count Before Dropping Duplicates : {df_bronze.count()}')
df_silver = df_bronze.dropDuplicates(['customer_id'])
print(f'Row Count After Dropping Duplicates : {df_silver.count()}')

In [0]:
display(
    df_silver.filter(F.col('customer_name') != F.trim(F.col('customer_name')))
)

In [0]:
df_silver = df_silver.withColumn(
    'customer_name',
    F.trim(F.col('customer_name'))
)

In [0]:
display(
    df_silver.filter(F.col('customer_name') != F.trim(F.col('customer_name')))
)

In [0]:
df_silver.select('city').distinct().show()

In [0]:
city_mapping = {
'Bengaluruu':'Bengalur',
'Bengalore':'Bengaluru',

'Hyderabadd': 'Hyderabad',
'Hyderbad': 'Hyderabad',

'NewDelhi': 'New Delhi',
'NewDheli': 'New Delhi',
'NewDelhee': 'New Delhi'
}

allowed = ['Bengaluru', 'Hyderabad', 'New Delhi']

df_silver = (df_silver
    .replace(city_mapping, subset = ['city'])
    .withColumn(
        'city', 
        F.when(F.col('city').isNull(), None)
        .when(F.col('city').isin(allowed), F.col('city'))
        .otherwise(None) 
    )
    )

df_silver.select('city').distinct().show() 

In [0]:
df_silver = df_silver.withColumn(
    'customer_name', 
    F.when(F.col("customer_name").isNull(), None)
    .otherwise(F.initcap('customer_name'))
)

df_silver.display()

In [0]:
df_silver.filter(F.col('city').isNull()).show(truncate = False)

In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col('customer_name').isin(null_customer_names)).show(truncate = False)

In [0]:
# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",
    
    # Zenathlete Foods
    789420: "Bengaluru",
    
    # Primefuel Nutrition
    789520: "Bengaluru",

    789521: "Hyderabad",
    
    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ["customer_id", 'fixed_city']
)

display(df_fix)

In [0]:
df_silver = (
    df_silver
    .join(df_fix, 'customer_id', 'left')
    .withColumn(
        'city',
        F.coalesce('city', 'fixed_city') 
    )
    .drop('fixed_city')
)

display(df_silver)

In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col('customer_name').isin(null_customer_names)).show(truncate = False)

In [0]:
df_silver = df_silver.withColumn(
    'customer_id',
    F.col('customer_id').cast('string')
)

print(df_silver.printSchema())

In [0]:
df_silver = (df_silver
        .withColumn(
        'customer', 
        F.concat_ws('-', 'customer_name', F.coalesce(F.col('city'), F.lit('Unknown')))
    )

    .withColumn('market', F.lit('India'))
    .withColumn('platform', F.lit('Sports Bar'))
    .withColumn('channel', F.lit('Acquisition'))
)

display(df_silver.limit(10))

In [0]:
(df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', 'True')
    .option('mergeSchema', 'True')
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

)